# 03_experiments — Model Experimentation & Tuning
**Week 3 | Telco Customer Churn**

Covers:
1. Baseline recap (Logistic Regression)
2. Random Forest — Optuna tuning
3. XGBoost — Optuna tuning
4. LightGBM — Optuna tuning
5. Stratified K-Fold cross-validation on best model
6. MLflow experiment log
7. Best model selection with rationale
8. Save best model

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import mlflow
import mlflow.sklearn
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb

from src.features import get_splits, get_preprocessor_pipeline, apply_smote
from src.evaluate import evaluate, print_report

RANDOM_STATE = 42
N_TRIALS     = 30   # Optuna trials per model (increase for better tuning)
CV_FOLDS     = 5

## 1. Load & Preprocess Data

In [ ]:
X_train, X_test, y_train, y_test = get_splits(
    path='../data/processed/telco_churn_cleaned.csv'
)

preprocessor = get_preprocessor_pipeline()
X_train_t = preprocessor.fit_transform(X_train)
X_test_t  = preprocessor.transform(X_test)

# SMOTE-balanced training set (used for tree models)
X_train_sm, y_train_sm = apply_smote(X_train_t, y_train, random_state=RANDOM_STATE)

print(f'Train (raw):   {X_train_t.shape}  | Churn: {y_train.mean():.2%}')
print(f'Train (SMOTE): {X_train_sm.shape} | Churn: {y_train_sm.mean():.2%}')
print(f'Test:          {X_test_t.shape}   | Churn: {y_test.mean():.2%}')

## 2. MLflow Setup

In [ ]:
mlflow.set_tracking_uri('../mlruns')
EXPERIMENT_NAME = 'telco_churn_week3'
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'MLflow experiment: {EXPERIMENT_NAME}')
print('Run: mlflow ui --backend-store-uri ../mlruns  to view results')

## 3. Helper — log a run to MLflow

In [ ]:
def log_run(model, params: dict, metrics: dict, run_name: str):
    """Log params, metrics, and model artifact to MLflow."""
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics({k: v for k, v in metrics.items() if k != 'label'})
        mlflow.sklearn.log_model(model, artifact_path='model')
    print(f'[MLflow] logged run: {run_name}  F1={metrics["f1"]}  AUC={metrics["roc_auc"]}')

## 4. Baseline Recap — Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_t, y_train)

lr_pred = lr.predict(X_test_t)
lr_prob = lr.predict_proba(X_test_t)[:, 1]
lr_metrics = evaluate(y_test, lr_pred, lr_prob, label='LR baseline')

log_run(lr, {'model': 'LogisticRegression', 'class_weight': 'balanced'}, lr_metrics, 'LR_baseline')
print(lr_metrics)

## 5. Random Forest — Optuna Tuning

In [ ]:
def rf_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600, step=100),
        'max_depth':         trial.suggest_int('max_depth', 4, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
        'class_weight':      'balanced',
        'random_state':      RANDOM_STATE,
        'n_jobs':            -1,
    }
    model = RandomForestClassifier(**params)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_t, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

rf_study = optuna.create_study(direction='maximize', study_name='random_forest')
rf_study.optimize(rf_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest RF params: {rf_study.best_params}')
print(f'Best CV ROC-AUC: {rf_study.best_value:.4f}')

In [ ]:
# Retrain best RF on full training set
best_rf = RandomForestClassifier(
    **rf_study.best_params,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
best_rf.fit(X_train_t, y_train)

rf_pred = best_rf.predict(X_test_t)
rf_prob = best_rf.predict_proba(X_test_t)[:, 1]
rf_metrics = evaluate(y_test, rf_pred, rf_prob, label='Random Forest')

log_run(best_rf, rf_study.best_params, rf_metrics, 'RandomForest_best')
print_report(y_test, rf_pred, 'Random Forest')
print(rf_metrics)

## 6. XGBoost — Optuna Tuning

In [ ]:
# Class imbalance ratio for scale_pos_weight
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600, step=100),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': neg_pos_ratio,
        'use_label_encoder': False,
        'eval_metric':      'logloss',
        'random_state':     RANDOM_STATE,
        'n_jobs':           -1,
    }
    model = xgb.XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_t, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction='maximize', study_name='xgboost')
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest XGB params: {xgb_study.best_params}')
print(f'Best CV ROC-AUC: {xgb_study.best_value:.4f}')

In [ ]:
best_xgb = xgb.XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=neg_pos_ratio,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
best_xgb.fit(X_train_t, y_train)

xgb_pred = best_xgb.predict(X_test_t)
xgb_prob = best_xgb.predict_proba(X_test_t)[:, 1]
xgb_metrics = evaluate(y_test, xgb_pred, xgb_prob, label='XGBoost')

log_run(best_xgb, xgb_study.best_params, xgb_metrics, 'XGBoost_best')
print_report(y_test, xgb_pred, 'XGBoost')
print(xgb_metrics)

## 7. LightGBM — Optuna Tuning

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600, step=100),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 150),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 50),
        'is_unbalance':     True,
        'random_state':     RANDOM_STATE,
        'n_jobs':           -1,
        'verbose':          -1,
    }
    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_t, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

lgb_study = optuna.create_study(direction='maximize', study_name='lightgbm')
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest LGB params: {lgb_study.best_params}')
print(f'Best CV ROC-AUC: {lgb_study.best_value:.4f}')

In [ ]:
best_lgb = lgb.LGBMClassifier(
    **lgb_study.best_params,
    is_unbalance=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
best_lgb.fit(X_train_t, y_train)

lgb_pred = best_lgb.predict(X_test_t)
lgb_prob = best_lgb.predict_proba(X_test_t)[:, 1]
lgb_metrics = evaluate(y_test, lgb_pred, lgb_prob, label='LightGBM')

log_run(best_lgb, lgb_study.best_params, lgb_metrics, 'LightGBM_best')
print_report(y_test, lgb_pred, 'LightGBM')
print(lgb_metrics)

## 8. Benchmark Table

In [ ]:
results = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics, lgb_metrics])
results = results.sort_values('roc_auc', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

## 9. Stratified K-Fold CV on Best Model

In [ ]:
# Identify best model by ROC-AUC
best_row = results.iloc[0]
print(f'Best model: {best_row["label"]}  ROC-AUC={best_row["roc_auc"]}')

model_map = {
    'LR baseline':  lr,
    'Random Forest': best_rf,
    'XGBoost':       best_xgb,
    'LightGBM':      best_lgb,
}
champion = model_map[best_row['label']]

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(champion, X_train_t, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_f1  = cross_val_score(champion, X_train_t, y_train, cv=cv, scoring='f1',      n_jobs=-1)

print(f'\n{CV_FOLDS}-Fold Stratified CV on {best_row["label"]}')
print(f'  ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  F1:      {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}')

## 10. Save Champion Model

In [ ]:
with open('../models/churn_model.pkl', 'wb') as f:
    pickle.dump({'model': champion, 'preprocessor': preprocessor}, f)

print(f'Champion model saved → models/churn_model.pkl')
print(f'Model type: {type(champion).__name__}')
print(f'Test ROC-AUC: {best_row["roc_auc"]}  |  F1: {best_row["f1"]}')